In [1]:

import config as cfg
import misc
import os
import torch
from utils import losses
from misc import get_model_summary
from data_preproc.data_preproc_functions import Logger
logger=Logger()

temp_folder = 'temp'


/home/d.c.macrae@mydre.org/miniconda3/envs/HNC_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import utils.load_data as load_data
import config as cfg
from data_preproc.data_preproc_functions import copy_file, copy_folder, create_folder_if_not_exists, Logger

logger=Logger(None, False)


train_dict, val_dict, test_dict = load_data.get_files(config=cfg, logger=logger)

(train_dl, val_dl, test_dl, batch_size, channels, depth, height, width, 
            n_features, train_dict, val_dict, test_dict) = \
            load_data.main(config=cfg, train_dict=train_dict, val_dict=val_dict, test_dict=test_dict,
                           fold=0, drop_last=cfg.dataloader_drop_last, logger=logger)

2024-05-13 07:35:02 - INFO: >>> FULL <<<
2024-05-13 07:35:02 - INFO: Expected size (fraction): 1090 (100%)
2024-05-13 07:35:02 - INFO: Size (fraction): 1090 (100.0%)
2024-05-13 07:35:02 - INFO: Group: CT+C
2024-05-13 07:35:02 - INFO: 	Value = 0, 	Proportion = 17.982% (196)
2024-05-13 07:35:02 - INFO: 	Value = 1, 	Proportion = 82.018% (894)
2024-05-13 07:35:02 - INFO: 
2024-05-13 07:35:02 - INFO: Group: CT_Artefact
2024-05-13 07:35:02 - INFO: 	Value = 0.0, 	Proportion = 80.459% (877)
2024-05-13 07:35:02 - INFO: 	Value = 1.0, 	Proportion = 19.541% (213)
2024-05-13 07:35:02 - INFO: 
2024-05-13 07:35:02 - INFO: Group: Photons
2024-05-13 07:35:02 - INFO: 	Value = 0.0, 	Proportion = 14.22% (155)
2024-05-13 07:35:02 - INFO: 	Value = 1.0, 	Proportion = 85.78% (935)
2024-05-13 07:35:02 - INFO: 
2024-05-13 07:35:02 - INFO: Group: Loctum2
2024-05-13 07:35:02 - INFO: 	Value = Overig, 	Proportion = 1.009% (11)
2024-05-13 07:35:02 - INFO: 	Value = Larynx, 	Proportion = 43.761% (477)
2024-05-13 07:35

In [8]:
device = cfg.device

for batch_num, batch_data in enumerate(train_dl):
    # send to device (the data is augmented within the train dataloader)
    train_inputs, train_features, train_label_list = (
        batch_data['ct_dose_seg'].to(device),
        batch_data['features'].to(device),
        batch_data['label_list'].to(device)
    )

    if batch_num == 0:
        break

In [9]:
train_label_list

tensor([[ 0,  0,  0,  0,  0],
        [ 0, -1,  0,  1,  1],
        [ 0,  1,  1, -1,  1],
        [ 0,  0,  1,  0,  1],
        [ 0,  0,  0,  0,  0],
        [ 0,  1,  0,  0,  1],
        [ 0,  0,  1,  0,  0],
        [ 0,  0,  1,  1,  1]], device='cuda:0')

In [10]:
train_inputs.shape

torch.Size([8, 3, 96, 96, 96])

In [11]:
channels = 3
depth = 96
height = 96
width = 96
n_features = len(cfg.features_dl)

cfg.dropout_p = 0.5
cfg.clinical_variables_position = 1
model, _ = misc.get_model(config=cfg, channels=channels, depth=depth, height=height, width=width, n_features=n_features,
                                pretrained_path=cfg.pretrained_path, logger=logger, save_summary = False)

# model2= misc.get_model(config=cfg, channels=channels, depth=depth, height=height, width=width, n_features=n_features,
#                                 pretrained_path=cfg.pretrained_path, logger=logger)

model.train()
# model2.eval()

model.to(device=cfg.device)
print('Model loaded')

/home/d.c.macrae@mydre.org/miniconda3/envs/HNC_env/lib/python3.11/site-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
2024-05-13 07:35:11 - INFO: Weight init name: kaiming_uniform.
2024-05-13 07:35:11 - INFO: Using our own weights init scheme for resnet_lrelu.


Layer (type (var_name))                                 Input Shape               Output Shape              Param #                   Kernel Shape
ResNet_LReLU (ResNet_LReLU)                             [8, 3, 96, 96, 96]        [8, 1]                    --                        --
├─Conv3d (conv1)                                        [8, 3, 96, 96, 96]        [8, 16, 48, 48, 48]       16,464                    [7, 7, 7]
├─BatchNorm3d (bn1)                                     [8, 16, 48, 48, 48]       [8, 16, 48, 48, 48]       32                        --
├─LeakyReLU (act)                                       [8, 16, 48, 48, 48]       [8, 16, 48, 48, 48]       --                        --
├─MaxPool3d (maxpool)                                   [8, 16, 48, 48, 48]       [8, 16, 24, 24, 24]       --                        3
├─ModuleList (backbone)                                 --                        --                        --                        --
│    └─Sequential (block0

train_inputs, train_features, train_label_list

In [12]:
loss_function = losses.get_loss_function(config=cfg, train_dict=train_dict, val_dict=val_dict)

In [13]:
device = cfg.device
endpoint_list = cfg.endpoint_list
N_passes = 10
all_preds = []
mean_predictions_dict = {endpoint: 0 for endpoint in cfg.endpoint_list}
valid_endpoints_as_tensor = torch.as_tensor(cfg.valid_endpoint_values, device=device)

train_y_pred_dict = dict()
train_y_true_dict = dict()
sub_ensemble_pred_dict = dict()
sub_ensemble_true_dict = dict()
mixup_label_indexes = torch.as_tensor([], dtype=torch.int32, device=device)
mixup_lambda_values = torch.as_tensor([], dtype=torch.float32, device=device)
for endpoint in endpoint_list:
    train_y_pred_dict[endpoint] = torch.as_tensor([], dtype=torch.float32, device=device)
    train_y_true_dict[endpoint] = torch.as_tensor([], dtype=torch.int8, device=device)
    sub_ensemble_pred_dict[endpoint] = torch.as_tensor([], dtype=torch.float32, device=device)
    sub_ensemble_true_dict[endpoint] = torch.as_tensor([], dtype=torch.int8, device=device)

mixup_indexes_batch, mixup_lambda = None, None


train_label_dict = dict()
for j, (endpoint, train_label) in enumerate(zip(endpoint_list, torch.swapaxes(train_label_list, 0, 1))):
    train_label_dict[endpoint] = train_label      

backbone_features = model.forward_backbone(x=train_inputs, features=train_features)

for i in range(N_passes):
        
    preds_i = model.forward_linear_layers(x=backbone_features, features=train_features)
    all_preds.append(preds_i)

    # keep track of the mean prediction of the forward passes on this batch
    mean_predictions_dict = misc.sum_dictionaries(mean_predictions_dict, preds_i, divisor=N_passes)

    # append this sub-ensemble prediction to the ensemble predictions for this batch
    for endpoint in cfg.endpoint_list:
        sub_ensemble_pred_dict[endpoint] = torch.cat([sub_ensemble_pred_dict[endpoint], preds_i[endpoint]], dim=0)
        sub_ensemble_true_dict[endpoint] = torch.cat([sub_ensemble_true_dict[endpoint], train_label_dict[endpoint]], dim=0)


# calculate the loss for this batch    
batch_loss, batch_per_toxicity_loss_dict, \
            unweighted_batch_loss, unweighted_batch_per_toxicity_loss_dict =  losses.calculate_loss(cfg, sub_ensemble_pred_dict, 
                                                                                                    sub_ensemble_true_dict, valid_endpoints_as_tensor, 
                                                                                                    loss_function, mixup_indexes_batch, mixup_lambda)

# stick the mean predictions into the epoch predictions
for endpoint in cfg.endpoint_list:
    train_y_pred_dict[endpoint] = torch.cat([train_y_pred_dict[endpoint], preds_i[endpoint]], dim=0)
    train_y_true_dict[endpoint] = torch.cat([train_y_true_dict[endpoint], train_label_dict[endpoint]], dim=0)

In [15]:
for endpoint in cfg.endpoint_list:
    print(sub_ensemble_true_dict[endpoint].shape)

torch.Size([80])
torch.Size([80])
torch.Size([80])
torch.Size([80])
torch.Size([80])


In [74]:
sub_ensemble_pred_dict['Aspiration_M06'].shape

torch.Size([80, 1])

In [68]:
batch_per_toxicity_loss_dict

{'Aspiration_M06': 0.7145906686782837,
 'Dysphagia_M06': 0.4178192615509033,
 'Sticky_M06': 0.41642361879348755,
 'Taste_M06': 0.5978848338127136,
 'Xerostomia_M06': 0.390950471162796}

In [69]:
batch_loss

tensor(0.5075, device='cuda:0', grad_fn=<ClampBackward1>)

In [70]:
# calculate the mean loss 
batch_loss, batch_per_toxicity_loss_dict, \
            unweighted_batch_loss, unweighted_batch_per_toxicity_loss_dict =  losses.calculate_loss(cfg, mean_predictions_dict, 
                                                                                                    train_label_dict, valid_endpoints_as_tensor, 
                                                                                                    loss_function, mixup_indexes_batch, mixup_lambda)

In [71]:
batch_per_toxicity_loss_dict

{'Aspiration_M06': 0.7138321399688721,
 'Dysphagia_M06': 0.41281425952911377,
 'Sticky_M06': 0.4142720103263855,
 'Taste_M06': 0.5974202156066895,
 'Xerostomia_M06': 0.39245879650115967}

In [72]:
batch_loss

tensor(0.5062, device='cuda:0', grad_fn=<ClampBackward1>)